# ROGII Wellbore Geology — LightGBM Baseline (Residual on Persistence)

## Abstract

This notebook implements a baseline solution for the ROGII Wellbore Geology Prediction competition. The model predicts a *residual* on top of a persistence baseline: for every row in the evaluation zone, the prediction is the last visible value of `TVT_input` plus a learned correction. The correction is produced by a LightGBM regressor trained on per-row features derived from the horizontal-well measurements, the typewell reference log, and statistics of the visible segment.

The approach is motivated by three observations established in the accompanying EDA notebook:

1. The target `TVT` is extremely smooth along measured depth (lag-1 autocorrelation ≈ 0.998).
2. The evaluation zone covers 73% of the well on average, making long-range linear extrapolation unstable: in the EDA, a 200-row linear fit produced an overall RMSE of 116 ft versus 16 ft for naive persistence.
3. The horizontal gamma-ray log is correlated with the typewell gamma-ray log (median Pearson r ≈ 0.77 across wells), but the correlation is not high enough to dominate persistence alone.

The model therefore takes persistence as the anchor and uses gamma-ray and trajectory features to learn a correction for the systematic dip and drift that persistence ignores.

The notebook is organised as follows:
1. Setup
2. Helper functions
3. Feature engineering
4. Training matrix construction
5. Test matrix construction
6. Cross-validated training
7. Out-of-fold evaluation
8. Submission generation
9. Notes on extension and on splitting into separate training and inference notebooks

## 1. Setup

In [ ]:
from pathlib import Path
from collections import defaultdict
import gc, warnings, time

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import lightgbm as lgb
from sklearn.model_selection import GroupKFold

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 100)
pd.set_option("display.max_colwidth", 120)

RNG = np.random.default_rng(42)

N_FOLDS = 5
LGB_PARAMS = dict(
    objective="regression",
    metric="rmse",
    learning_rate=0.03,
    num_leaves=63,
    min_data_in_leaf=200,
    feature_fraction=0.85,
    bagging_fraction=0.85,
    bagging_freq=5,
    lambda_l2=1.0,
    verbose=-1,
    seed=42,
)
NUM_BOOST_ROUND = 2000
EARLY_STOPPING = 100

## 2. Path setup and helpers

In [ ]:
DATA_ROOT = Path("/kaggle/input/competitions/rogii-wellbore-geology-prediction")
TRAIN_DIR = DATA_ROOT / "train"
TEST_DIR  = DATA_ROOT / "test"
SAMPLE_SUB = DATA_ROOT / "sample_submission.csv"
print("DATA_ROOT:", DATA_ROOT)
print("Train dir contents (head):", sorted(p.name for p in TRAIN_DIR.iterdir())[:5])
print("Test  dir contents (head):", sorted(p.name for p in TEST_DIR.iterdir())[:5])

In [ ]:
def well_id_from(path: Path) -> str:
    return path.name.split("__")[0].replace(".png", "")

def list_wells(split_dir: Path):
    return [well_id_from(p) for p in sorted(split_dir.glob("*__horizontal_well.csv"))]

def horiz_path(wid, split):
    return (TRAIN_DIR if split == "train" else TEST_DIR) / f"{wid}__horizontal_well.csv"

def type_path(wid, split):
    return (TRAIN_DIR if split == "train" else TEST_DIR) / f"{wid}__typewell.csv"

train_wells = list_wells(TRAIN_DIR)
test_wells  = list_wells(TEST_DIR)
print(f"Training wells: {len(train_wells)}")
print(f"Test wells:     {len(test_wells)}")

In [ ]:
def robust_slope(x, y, default=0.0):
    """Slope of y on x via least squares; returns default on degenerate input."""
    x = np.asarray(x, dtype=float); y = np.asarray(y, dtype=float)
    m = np.isfinite(x) & np.isfinite(y)
    if m.sum() < 2: return default
    xx = x[m] - x[m].mean()
    yy = y[m] - y[m].mean()
    denom = float(np.sum(xx * xx))
    if denom <= 1e-12: return default
    return float(np.sum(xx * yy) / denom)

def first_nan_idx(s: pd.Series):
    mask = s.isna()
    if not mask.any(): return None
    return int(mask.idxmax()) if mask.iloc[0] else int(np.argmax(mask.values))

## 3. Feature engineering

For every well, the rows of the evaluation zone are converted into feature rows. Two families of features are produced:

**Per-well static features** (constant across all evaluation rows of a single well):
- last visible values of `TVT_input`, `MD`, `X`, `Y`, `Z`, `GR`;
- summary statistics of the visible segment (mean, std, range of `TVT_input` and `GR`);
- slopes of `TVT_input` against `MD` over multiple windows (50, 200, 500 rows, full visible segment);
- typewell summary statistics (`TVT` range, `GR` median and standard deviation);
- gamma-ray alignment parameters estimated from the visible segment alone: the optimal vertical shift Δ that maximises the correlation between horizontal `GR(MD)` and typewell `GR(TVT − Δ)`, the resulting Pearson correlation, and the mean horizontal-minus-typewell bias.

**Per-row dynamic features** (vary across the evaluation rows of a single well):
- raw `MD`, `X`, `Y`, `Z`, `GR`;
- offsets from the prediction-start row: `MD − last_MD`, `Z − last_Z`, lateral distance, row-index difference, fractional row position;
- per-well-normalised gamma ray;
- the difference between the row's gamma ray and the typewell gamma ray sampled at `last_known_TVT − Δ` (a fixed depth that approximates the row's expected stratigraphic position);
- the typewell `TVT` whose gamma-ray reading best matches the row's gamma ray within a ±150 ft window of `last_known_TVT`, and its offset from `last_known_TVT`;
- centred rolling gamma-ray statistics (mean, standard deviation) over the eval-zone gamma-ray sequence.

The target is the residual `TVT − last_known_TVT`. At inference, the predicted `TVT` is recovered as `last_known_TVT + predicted_residual`.

In [ ]:
def estimate_alignment_from_visible(hw_visible, tw):
    """Estimate the typewell shift Δ, correlation r, and bias from the visible portion.

    Uses only rows where TVT_input is present, so this is computable at inference time.
    Returns dict with keys: shift_ft, corr, bias."""
    out = {"shift_ft": 0.0, "corr": np.nan, "bias": 0.0}
    if tw is None or "TVT" not in tw or "GR" not in tw or "GR" not in hw_visible:
        return out
    tw_clean = tw.dropna(subset=["TVT", "GR"]).sort_values("TVT")
    if len(tw_clean) < 10: return out
    sub = hw_visible.dropna(subset=["TVT_input", "GR"])
    if len(sub) < 50: return out
    tvt = sub["TVT_input"].values
    gr_h = sub["GR"].values
    best_r = -np.inf; best_s = 0.0
    shifts = np.arange(-30, 31, 2)
    for s in shifts:
        g = np.interp(tvt + s, tw_clean["TVT"].values, tw_clean["GR"].values,
                      left=np.nan, right=np.nan)
        v = np.isfinite(g) & np.isfinite(gr_h)
        if v.sum() < 30: continue
        rs = float(np.corrcoef(gr_h[v], g[v])[0, 1])
        if np.isfinite(rs) and rs > best_r:
            best_r = rs; best_s = float(s)
    # Bias at best shift
    if np.isfinite(best_r):
        g = np.interp(tvt + best_s, tw_clean["TVT"].values, tw_clean["GR"].values,
                      left=np.nan, right=np.nan)
        v = np.isfinite(g) & np.isfinite(gr_h)
        out["shift_ft"] = best_s
        out["corr"] = best_r
        out["bias"] = float(np.nanmean(gr_h[v] - g[v])) if v.any() else 0.0
    return out

In [ ]:
def build_features_for_well(wid, split, test_eval_idx=None):
    """Construct the feature matrix for one well's evaluation rows.

    Args:
        wid: well id.
        split: 'train' or 'test'.
        test_eval_idx: for test wells, set of row indices required by the submission.
                       If None, defaults to all TVT_input == NaN rows.
    Returns: DataFrame with one row per evaluation row, including 'id', 'well_id',
             'row_index', features, and (for train) 'target_residual' and 'target_tvt'."""
    h = pd.read_csv(horiz_path(wid, split))
    h["row_index"] = np.arange(len(h), dtype=np.int64)
    if "TVT_input" not in h.columns:
        return pd.DataFrame()
    
    # Identify evaluation rows.
    eval_mask = h["TVT_input"].isna().values
    if not eval_mask.any():
        return pd.DataFrame()
    
    if split == "train":
        if "TVT" not in h.columns:
            return pd.DataFrame()
        # Use rows where the target is known and TVT_input is hidden.
        sel_mask = eval_mask & h["TVT"].notna().values
    else:
        sel_mask = eval_mask
        if test_eval_idx is not None:
            mask2 = np.zeros(len(h), dtype=bool)
            mask2[list(test_eval_idx)] = True
            sel_mask = sel_mask & mask2
    
    if not sel_mask.any():
        return pd.DataFrame()
    
    visible = h[h["TVT_input"].notna()].copy()
    if len(visible) == 0: return pd.DataFrame()
    
    last = visible.iloc[-1]
    last_TVT = float(last["TVT_input"])
    last_MD  = float(last["MD"])
    last_X   = float(last["X"])
    last_Y   = float(last["Y"])
    last_Z   = float(last["Z"])
    last_GR  = float(last["GR"]) if "GR" in last and pd.notna(last["GR"]) else np.nan
    
    ps_idx = first_nan_idx(h["TVT_input"])
    
    # Visible-segment statistics.
    vis_TVT = visible["TVT_input"].values
    vis_GR  = visible["GR"].values if "GR" in visible.columns else np.array([])
    vis_n = len(visible)
    
    # Slopes of TVT vs MD over multiple windows.
    slope_all = robust_slope(visible["MD"].values, vis_TVT)
    def slope_window(K):
        if vis_n < 2: return slope_all
        return robust_slope(visible["MD"].values[-min(K, vis_n):],
                            vis_TVT[-min(K, vis_n):], default=slope_all)
    slope_K50  = slope_window(50)
    slope_K200 = slope_window(200)
    slope_K500 = slope_window(500)
    slope_TVT_Z = robust_slope(visible["Z"].values[-min(200, vis_n):],
                               vis_TVT[-min(200, vis_n):])
    
    # Typewell load and alignment.
    tp = type_path(wid, split)
    tw = pd.read_csv(tp) if tp.is_file() else None
    align = estimate_alignment_from_visible(visible, tw)
    
    # Typewell summary statistics.
    if tw is not None and len(tw):
        tw_TVT_min = float(tw["TVT"].min()) if "TVT" in tw else np.nan
        tw_TVT_max = float(tw["TVT"].max()) if "TVT" in tw else np.nan
        tw_GR_med  = float(tw["GR"].median()) if "GR" in tw else np.nan
        tw_GR_std  = float(tw["GR"].std()) if "GR" in tw else np.nan
        tw_clean = tw.dropna(subset=["TVT", "GR"]).sort_values("TVT") if ("TVT" in tw and "GR" in tw) else None
    else:
        tw_TVT_min = tw_TVT_max = tw_GR_med = tw_GR_std = np.nan
        tw_clean = None
    
    # Per-well GR normalisation parameters.
    pw_gr_med = float(np.nanmedian(vis_GR)) if len(vis_GR) else np.nan
    pw_gr_std = float(np.nanstd(vis_GR)) if len(vis_GR) else np.nan
    if not (pw_gr_std > 0): pw_gr_std = 1.0
    
    # Build per-row feature frame.
    sel_idx = np.flatnonzero(sel_mask)
    cur = h.iloc[sel_idx].copy()
    cur["well_id"] = wid
    cur["id"] = cur["well_id"] + "_" + cur["row_index"].astype(str)
    
    # Static per-well features (broadcast).
    static = {
        "last_known_TVT": last_TVT,
        "last_known_MD": last_MD, "last_known_X": last_X, "last_known_Y": last_Y,
        "last_known_Z": last_Z, "last_known_GR": last_GR,
        "vis_n_rows": vis_n,
        "vis_md_range": float(visible["MD"].max() - visible["MD"].min()),
        "vis_TVT_min": float(np.nanmin(vis_TVT)),
        "vis_TVT_max": float(np.nanmax(vis_TVT)),
        "vis_TVT_range": float(np.nanmax(vis_TVT) - np.nanmin(vis_TVT)),
        "vis_TVT_mean": float(np.nanmean(vis_TVT)),
        "vis_TVT_std":  float(np.nanstd(vis_TVT)),
        "vis_TVT_first": float(vis_TVT[0]),
        "vis_GR_mean":  float(np.nanmean(vis_GR)) if len(vis_GR) else np.nan,
        "vis_GR_std":   pw_gr_std,
        "vis_GR_median": pw_gr_med,
        "vis_GR_min":   float(np.nanmin(vis_GR)) if len(vis_GR) else np.nan,
        "vis_GR_max":   float(np.nanmax(vis_GR)) if len(vis_GR) else np.nan,
        "slope_TVT_MD_all":  slope_all,
        "slope_TVT_MD_K50":  slope_K50,
        "slope_TVT_MD_K200": slope_K200,
        "slope_TVT_MD_K500": slope_K500,
        "slope_TVT_Z_K200":  slope_TVT_Z,
        "tw_TVT_min": tw_TVT_min, "tw_TVT_max": tw_TVT_max,
        "tw_GR_med": tw_GR_med, "tw_GR_std": tw_GR_std,
        "align_shift_ft": align["shift_ft"],
        "align_corr": align["corr"],
        "align_bias": align["bias"],
        "ps_row_idx": ps_idx if ps_idx is not None else len(h),
        "n_total_rows": len(h),
    }
    for k, v in static.items():
        cur[k] = v
    
    # Per-row dynamic features.
    cur["md_from_ps"] = cur["MD"].values - last_MD
    cur["z_from_ps"]  = cur["Z"].values - last_Z
    cur["dxy_from_ps"] = np.sqrt((cur["X"].values - last_X)**2 +
                                  (cur["Y"].values - last_Y)**2)
    cur["row_from_ps"] = cur["row_index"].values - (ps_idx if ps_idx is not None else 0)
    cur["row_frac"]    = cur["row_index"].values / max(len(h) - 1, 1)
    cur["GR_norm"]     = (cur["GR"].values - pw_gr_med) / pw_gr_std
    
    # GR comparison with typewell at last_known_TVT − Δ (a fixed depth proxy).
    if tw_clean is not None and len(tw_clean):
        gr_at_last = float(np.interp(last_TVT - align["shift_ft"],
                                      tw_clean["TVT"].values, tw_clean["GR"].values,
                                      left=np.nan, right=np.nan))
        cur["gr_minus_tw_at_last"] = cur["GR"].values - gr_at_last - align["bias"]
        # Best-match TVT within ±150 ft of last_known_TVT.
        win = tw_clean[(tw_clean["TVT"] >= last_TVT - 150) &
                       (tw_clean["TVT"] <= last_TVT + 150)]
        if len(win) >= 5:
            tvts = win["TVT"].values
            grs  = win["GR"].values
            grs_db = grs + align["bias"]  # adjust typewell GR for bias
            best_match = np.empty(len(cur))
            for i, g in enumerate(cur["GR"].values):
                if not np.isfinite(g):
                    best_match[i] = last_TVT
                else:
                    j = int(np.argmin(np.abs(grs_db - g)))
                    best_match[i] = tvts[j]
            cur["best_match_tvt"] = best_match
            cur["best_match_delta"] = best_match - last_TVT
        else:
            cur["best_match_tvt"] = last_TVT
            cur["best_match_delta"] = 0.0
    else:
        cur["gr_minus_tw_at_last"] = np.nan
        cur["best_match_tvt"] = last_TVT
        cur["best_match_delta"] = 0.0
    
    # Centred rolling GR statistics over all rows of the well (uses eval-zone GR too,
    # which is available at inference). Window = 25 rows.
    gr_full = h["GR"].values
    gr_series = pd.Series(gr_full).interpolate(limit_direction="both")
    roll_mean = gr_series.rolling(25, center=True, min_periods=1).mean().values
    roll_std  = gr_series.rolling(25, center=True, min_periods=1).std().fillna(0.0).values
    cur["roll_GR_mean25"] = roll_mean[sel_idx]
    cur["roll_GR_std25"]  = roll_std[sel_idx]
    cur["GR_minus_rollmean"] = cur["GR"].values - cur["roll_GR_mean25"].values
    
    # Persistence-anchor target.
    if split == "train":
        cur["target_tvt"] = h["TVT"].values[sel_idx]
        cur["target_residual"] = cur["target_tvt"].values - last_TVT
    
    return cur.reset_index(drop=True)

## 4. Build training matrix

In [ ]:
t0 = time.time()
train_parts = []
for k, wid in enumerate(train_wells):
    df = build_features_for_well(wid, "train")
    if len(df):
        train_parts.append(df)
    if (k + 1) % 100 == 0:
        print(f"  built features for {k+1}/{len(train_wells)} wells "
              f"(rows so far: {sum(len(p) for p in train_parts):,})")
train_df = pd.concat(train_parts, ignore_index=True)
del train_parts; gc.collect()
print(f"\nTrain feature matrix: {train_df.shape}  (built in {time.time()-t0:.1f}s)")
display(train_df.head(2))

In [ ]:
# Define the feature column list.
# Exclude IDs, targets, and any column not present at inference time.
# - TVT is the actual target (would be a direct leak in training).
# - TVT_input is all-NaN in selected rows by construction (eval-zone rows).
# - Formation columns (ANCC, ASTNU, ASTNL, EGFDU, EGFDL, BUDA) exist only in train.
exclude = {
    "well_id", "id", "row_index", "target_tvt", "target_residual",
    "TVT", "TVT_input",
    "ANCC", "ASTNU", "ASTNL", "EGFDU", "EGFDL", "BUDA",
}
feature_cols = [c for c in train_df.columns if c not in exclude]
print(f"Number of features: {len(feature_cols)}")
print("First 20 features:", feature_cols[:20])

# Cast to float32 for memory and speed.
X_train = train_df[feature_cols].astype(np.float32).values
y_train = train_df["target_residual"].astype(np.float32).values
groups  = train_df["well_id"].values
print(f"X shape: {X_train.shape}  y shape: {y_train.shape}")

## 5. Build test matrix

In [ ]:
sample_sub = pd.read_csv(SAMPLE_SUB)
sample_sub["well_id"] = sample_sub["id"].str.rsplit("_", n=1).str[0]
sample_sub["row_index"] = sample_sub["id"].str.rsplit("_", n=1).str[1].astype(int)
print(f"Sample submission shape: {sample_sub.shape}")
print(f"Distinct test wells:     {sample_sub['well_id'].nunique()}")

test_eval_index = (sample_sub.groupby("well_id")["row_index"]
                   .apply(lambda s: set(s.tolist()))
                   .to_dict())

t0 = time.time()
test_parts = []
for k, wid in enumerate(test_wells):
    df = build_features_for_well(wid, "test", test_eval_idx=test_eval_index.get(wid))
    if len(df):
        test_parts.append(df)
test_df = pd.concat(test_parts, ignore_index=True) if test_parts else pd.DataFrame()
print(f"\nTest feature matrix: {test_df.shape}  (built in {time.time()-t0:.1f}s)")

## 6. Cross-validated training

A 5-fold `GroupKFold` splits wells into folds; this prevents rows from the same well from appearing in both training and validation. The validation RMSE reported here is computed on the residual prediction added back to the persistence anchor, so it is directly comparable to the competition metric.

In [ ]:
gkf = GroupKFold(n_splits=N_FOLDS)
oof_residual = np.zeros(len(train_df), dtype=np.float64)
fold_models = []
fold_metrics = []

for fold, (tr_idx, va_idx) in enumerate(gkf.split(X_train, y_train, groups)):
    print(f"\n--- Fold {fold+1}/{N_FOLDS} ---")
    print(f"  train rows: {len(tr_idx):,}  val rows: {len(va_idx):,}  "
          f"train wells: {pd.Series(groups[tr_idx]).nunique()}  val wells: {pd.Series(groups[va_idx]).nunique()}")
    
    dtrain = lgb.Dataset(X_train[tr_idx], y_train[tr_idx], feature_name=feature_cols)
    dvalid = lgb.Dataset(X_train[va_idx], y_train[va_idx], feature_name=feature_cols,
                          reference=dtrain)
    booster = lgb.train(
        LGB_PARAMS, dtrain,
        num_boost_round=NUM_BOOST_ROUND,
        valid_sets=[dtrain, dvalid],
        valid_names=["train", "valid"],
        callbacks=[lgb.early_stopping(EARLY_STOPPING, verbose=False),
                   lgb.log_evaluation(period=200)],
    )
    pred_residual = booster.predict(X_train[va_idx], num_iteration=booster.best_iteration)
    oof_residual[va_idx] = pred_residual
    
    # Per-fold RMSE on TVT (residual + persistence anchor).
    y_true_tvt = train_df["target_tvt"].values[va_idx]
    last_tvt = train_df["last_known_TVT"].values[va_idx]
    pred_tvt = last_tvt + pred_residual
    rmse_residual_only = float(np.sqrt(np.mean((pred_residual - y_train[va_idx])**2)))
    rmse_tvt = float(np.sqrt(np.mean((pred_tvt - y_true_tvt)**2)))
    rmse_b0 = float(np.sqrt(np.mean((last_tvt - y_true_tvt)**2)))
    print(f"  fold RMSE  residual-only = {rmse_residual_only:.4f}")
    print(f"  fold RMSE  TVT (model)   = {rmse_tvt:.4f}")
    print(f"  fold RMSE  TVT (B0 only) = {rmse_b0:.4f}")
    
    fold_models.append(booster)
    fold_metrics.append({"fold": fold + 1, "best_iter": booster.best_iteration,
                          "rmse_residual": rmse_residual_only,
                          "rmse_tvt_model": rmse_tvt, "rmse_tvt_B0": rmse_b0})

print()
print("Per-fold summary:")
display(pd.DataFrame(fold_metrics).round(4))

In [ ]:
# Overall OOF metrics.
last_tvt_all = train_df["last_known_TVT"].values
oof_tvt = last_tvt_all + oof_residual
y_true_all = train_df["target_tvt"].values
rmse_oof_tvt = float(np.sqrt(np.mean((oof_tvt - y_true_all)**2)))
rmse_oof_b0  = float(np.sqrt(np.mean((last_tvt_all - y_true_all)**2)))
print(f"OOF RMSE — model: {rmse_oof_tvt:.4f}")
print(f"OOF RMSE — B0:    {rmse_oof_b0:.4f}")
print(f"Improvement over B0: {rmse_oof_b0 - rmse_oof_tvt:+.4f} ft "
      f"({(1 - rmse_oof_tvt / rmse_oof_b0) * 100:+.2f}%)")

## 7. Out-of-fold evaluation

This section examines per-well RMSE distribution, residual-prediction quality, and feature importance averaged over folds.

In [ ]:
# Per-well RMSE distribution.
per_well = (pd.DataFrame({"well_id": train_df["well_id"].values,
                           "y_true": y_true_all, "y_pred": oof_tvt,
                           "y_b0":   last_tvt_all})
            .groupby("well_id")
            .apply(lambda g: pd.Series({
                "rmse_model": float(np.sqrt(np.mean((g["y_pred"] - g["y_true"])**2))),
                "rmse_b0":    float(np.sqrt(np.mean((g["y_b0"]   - g["y_true"])**2))),
                "n":          len(g)}))
            .reset_index())
per_well["delta_vs_b0"] = per_well["rmse_b0"] - per_well["rmse_model"]
print("Per-well RMSE summary:")
display(per_well[["rmse_model", "rmse_b0", "delta_vs_b0", "n"]].describe(percentiles=[0.1, 0.5, 0.9]).round(3))
print(f"\nWells where the model beats B0: {(per_well['delta_vs_b0'] > 0).sum()} / {len(per_well)} "
      f"({(per_well['delta_vs_b0'] > 0).mean() * 100:.1f}%)")

In [ ]:
# Histogram and scatter.
fig, ax = plt.subplots(1, 2, figsize=(13, 4.5), constrained_layout=True)
ax[0].hist(per_well["delta_vs_b0"], bins=50, color="#1f77b4", edgecolor="white")
ax[0].axvline(0, color="black", lw=0.7, ls="--")
ax[0].set_title("Per-well RMSE improvement vs B0 (positive = model better)")
ax[0].set_xlabel("RMSE(B0) − RMSE(model)  [ft]")
ax[0].set_ylabel("number of wells")

ax[1].scatter(per_well["rmse_b0"], per_well["rmse_model"], s=8, alpha=0.5)
lims = [0, max(per_well["rmse_b0"].max(), per_well["rmse_model"].max())]
ax[1].plot(lims, lims, color="black", lw=0.7, ls="--", label="y = x")
ax[1].set_xlabel("RMSE — B0 (ft)")
ax[1].set_ylabel("RMSE — model (ft)")
ax[1].set_title("Per-well RMSE: model vs B0")
ax[1].legend()
plt.show()

In [ ]:
# Feature importance averaged over folds.
imp_arr = np.zeros(len(feature_cols))
for booster in fold_models:
    imp_arr += booster.feature_importance(importance_type="gain")
imp_arr /= len(fold_models)
imp_df = (pd.DataFrame({"feature": feature_cols, "gain": imp_arr})
          .sort_values("gain", ascending=False).reset_index(drop=True))
print("Top 25 features by mean gain:")
display(imp_df.head(25))

fig, ax = plt.subplots(figsize=(8, 8))
top = imp_df.head(25)[::-1]
ax.barh(top["feature"], top["gain"], color="#1f77b4")
ax.set_title("LightGBM feature importance — top 25 (mean gain over folds)")
ax.set_xlabel("gain")
plt.show()

## 8. Test prediction and submission

Predictions from the five fold models are averaged. The persistence anchor `last_known_TVT` is added back to the predicted residual to recover the predicted `TVT`. As a safety guard, the residual is clipped to ±400 ft, which exceeds the largest residual observed in the training set by a comfortable margin and prevents the model from emitting catastrophic outliers.

In [ ]:
if len(test_df):
    X_test = test_df[feature_cols].astype(np.float32).values
    fold_preds = np.zeros((len(test_df), len(fold_models)), dtype=np.float64)
    for j, booster in enumerate(fold_models):
        fold_preds[:, j] = booster.predict(X_test, num_iteration=booster.best_iteration)
    pred_residual = fold_preds.mean(axis=1)
    
    # Clip residual to a defensible range based on training data.
    residual_clip = 400.0
    pred_residual = np.clip(pred_residual, -residual_clip, residual_clip)
    
    pred_tvt = test_df["last_known_TVT"].values + pred_residual
    test_df["tvt_pred"] = pred_tvt
    print(f"Predicted residual stats — mean: {pred_residual.mean():+.3f}  "
          f"std: {pred_residual.std():.3f}  "
          f"min: {pred_residual.min():.3f}  max: {pred_residual.max():.3f}")
    print(f"Predicted TVT stats     — mean: {pred_tvt.mean():.1f}  "
          f"min: {pred_tvt.min():.1f}  max: {pred_tvt.max():.1f}")
else:
    print("Test feature matrix is empty.")

In [ ]:
# Build the submission file.
if len(test_df):
    pred_map = test_df.set_index("id")["tvt_pred"].to_dict()
    submission = sample_sub[["id"]].copy()
    submission["tvt"] = submission["id"].map(pred_map)
    
    # Fill any missing predictions with the well's last_known_TVT (defensive fallback).
    if submission["tvt"].isna().any():
        anchor_map = test_df.drop_duplicates("well_id").set_index("well_id")["last_known_TVT"].to_dict()
        miss = submission["tvt"].isna()
        submission.loc[miss, "tvt"] = (
            submission.loc[miss, "id"].str.rsplit("_", n=1).str[0].map(anchor_map)
        )
    submission["tvt"] = submission["tvt"].fillna(0.0)
    
    submission.to_csv("submission.csv", index=False)
    print(f"Submission written: shape = {submission.shape}")
    display(submission.head())
    print()
    print("tvt summary in submission:")
    print(submission["tvt"].describe().round(3))
else:
    # Defensive: still produce a valid submission file using last-known TVT only.
    submission = sample_sub[["id"]].copy()
    submission["tvt"] = 0.0
    submission.to_csv("submission.csv", index=False)
    print("Test set was empty; wrote zero-filled submission.")

## 9. Notes on extension

The baseline as presented is intentionally narrow in scope. Several directions are likely to yield further improvement and are worth pursuing:

**Better persistence anchor.** Instead of the single last visible value, a short rolling mean or a smoothed extrapolation that reverts to the visible mean over a few hundred feet may produce a more stable anchor with smaller residuals.

**Sequence-aware modelling.** A 1D convolutional network or a small transformer operating along measured depth, taking the visible `TVT_input` and the full `GR(MD)` sequence as input, can in principle propagate information across the eval zone in a way that row-independent gradient boosting cannot.

**Auxiliary targets in training.** The training files contain six geological surface columns (`ANCC`, `ASTNU`, `ASTNL`, `EGFDU`, `EGFDL`, `BUDA`) and the typewell `Geology` label. These are not available at inference, but they can be used as auxiliary regression targets in a multi-task setup. A model that learns to estimate the depth of these surfaces while predicting `TVT` may generalise more robustly.

**Spatial features.** With more training and test wells than the public stub provides, neighbour-based features (mean `TVT(MD)` across the K nearest wells in `(X, Y)`, dip estimates from neighbours) become viable. The current notebook leaves the spatial feature space largely unexploited.

**Stronger gamma-ray alignment.** The current implementation estimates a single global shift `Δ` per well. A locally varying alignment, estimated by dynamic time warping or by sliding-window correlation, may capture the depth-dependent dip that a constant shift cannot.

**Per-well rescaling.** The horizontal gamma-ray distribution varies substantially across wells. Robust per-well scaling (median-IQR rather than mean-std) before any GR-based feature is computed may stabilise the model.

---

## Notes on splitting into separate training and inference notebooks

For this LightGBM baseline, a single notebook is sufficient: the entire pipeline (feature engineering, training, inference, submission) finishes in under one hour on the standard Kaggle CPU runtime, well within the nine-hour competition limit.

Splitting into a training notebook and an inference notebook becomes appropriate when:

1. **Training time exceeds the competition runtime budget.** Deep models on the full training set can require many hours on GPU; in that case the model is trained in a separate notebook (or offline) and persisted as a Kaggle Dataset, and the submission notebook only loads the saved checkpoints and runs inference.
2. **The model is an ensemble of many independently trained components.** Training each component in a single notebook may be impractical; separate training runs save artefacts that the inference notebook combines.
3. **Inference iteration is slow because training is repeatedly redone.** Once the model is fixed, a separate inference notebook isolates submission-only work.

The mechanics of a split are:

1. Run the training notebook outside of the submission flow. Save the trained boosters (or whatever artefacts the model requires) using `booster.save_model(...)` for LightGBM, `joblib.dump(...)` for scikit-learn objects, or framework-specific checkpoint files for neural networks.
2. Upload the saved files as a private Kaggle Dataset.
3. In the inference notebook, attach the dataset, load the artefacts, run the feature-engineering and prediction code, and write `submission.csv`. The inference notebook must not require internet access or external resources beyond the attached datasets.
4. Confirm that the inference notebook's runtime, including dataset loading, fits inside the competition limit and produces a valid submission file in the working directory.

For the present baseline the split is unnecessary, but the codebase is structured so that the training cells (sections 4 and 6) and the inference cells (sections 5 and 8) can be separated with minimal modification when a future model justifies the additional complexity.